# QuetHoaDon · Tinh chỉnh VietOCR cho chữ số, OCR lại, M1b v4

Mục tiêu: giảm lỗi OCR ở **chữ số** (đơn giá, thành tiền, tổng tiền). VietOCR vòng 1 chỉ học dòng của 4 trường có nhãn.
Vòng này thêm (1) **chữ số tổng hợp** vẽ kiểu hóa đơn nhiệt (`src/recognition/synth_numbers.py`) và (2) **con số của các
hóa đơn train `verified`** (`src/recognition/number_pairs.py`), trộn với các cặp cũ để không quên chữ.

**Cách chạy:** GPU **T4**, Internet **On**. Gắn **Vietnamese Receipts MC_OCR 2021** và dataset `phase1_eval`
(cần `vietocr_ft/`, `tok_ft_n05/`, `eval/m1b_v3_val_probs.npz`). Chạy bằng **Save Version → Save & Run All**: chạy nền,
khoảng 3 giờ, mọi file trong `/kaggle/working` được lưu cùng phiên bản (tab Output). Chỉ train và validation; test không đọc.

**Cổng (viết trước khi chạy):**

1. **Cổng 1 (OCR):** trên vùng TOTAL_COST có nhãn của validation, tỷ lệ đúng chữ số **tăng** và CER không tăng; CER
   các vùng CRAFT của validation (bộ đo vòng 1) không tăng quá 0,2 điểm. Không đạt thì dừng sau cell 9.
2. **Cổng 2 (sản phẩm):** trên validation, so với M1b v3: tổng tiền đúng tuyệt đối (bật tổng từ các món) **tăng** và số
   hóa đơn sửa được nhiều hơn số bị làm hỏng; tổng các món = tổng chuẩn không giảm quá 1 hóa đơn.

In [ ]:
# CELL 1 - detectron2 (cho M1b ở cell 11; 3-10 phút). Mong đợi: "detectron2 OK"
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Mã nguồn nhánh Thinh/ocr-line-merge và thư viện OCR. Mong đợi: một dòng commit, "xong"
BRANCH = "Thinh/ocr-line-merge"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb 2>&1 | tail -2
print("xong")

In [ ]:
# CELL 3 [KIỂM THỬ] - GPU, dữ liệu, unit test. Mong đợi: "GPU: True Tesla T4", 5 đường dẫn, "OK"
import glob, os, sys, json, csv, torch
REPO, W = '/kaggle/working/QuetHoaDon', '/kaggle/working'
sys.path.insert(0, REPO); sys.path.insert(0, f'{REPO}/scripts')
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Không thấy GPU: Settings -> Accelerator -> GPU T4"
one = lambda pattern: sorted(glob.glob(pattern, recursive=True))[-1]
CSV = one('/kaggle/input/**/mcocr_train_df.csv')
IMG_RAW = os.path.dirname(one('/kaggle/input/**/train_images/**/*.jpg'))
VFT1 = one('/kaggle/input/**/vietocr_ft/vietocr_finetuned.pth')
TOKC = os.path.dirname(one('/kaggle/input/**/tok_ft_n05/validation.jsonl'))
PROBS3 = one('/kaggle/input/**/m1b_v3_val_probs.npz')
for k in ("CSV", "IMG_RAW", "VFT1", "TOKC", "PROBS3"):
    print(f"{k} = {globals()[k]}")
os.makedirs(f'{W}/eval', exist_ok=True)
!cd {REPO} && python -m unittest discover -s tests -t . 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"

In [ ]:
# CELL 4 [KIỂM THỬ] - File chia tập, phải trùng tok_ft_n05. Mong đợi: "train khớp: True", "validation khớp: True"
from src.training.data import load_jsonl
!python scripts/prepare_dataset.py --csv "{CSV}" --images "{IMG_RAW}" --out {W}/dataset > {W}/dataset_summary.txt
MANIFEST = f'{W}/dataset/mcocr_manifest_split.csv'
rows = list(csv.DictReader(open(MANIFEST, encoding='utf-8')))
for sp in ("train", "validation"):
    new = {r['img_id'] for r in rows if r['split'] == sp}
    old = {r['img_id'] for r in load_jsonl(os.path.join(TOKC, f'{sp}.jsonl'))}
    print(f"{sp} khớp:", new == old, f"({len(new)} hóa đơn)")

In [ ]:
# CELL 5 - OCR train + validation với VietOCR vòng 1 (khoảng 40 phút): ảnh tiền xử lý + JSON để cắt cặp huấn luyện.
# Mong đợi: "[done] {'ok': 1037, ...}"
COMMON = f'--images "{IMG_RAW}" --split-manifest {MANIFEST} --nested-threshold 0.5 --device cuda --splits train,validation'
OCR1 = f'{W}/ocr_r1'
!python -u scripts/run_ocr_dataset.py {COMMON} --weights {VFT1} --out {OCR1} 2>&1 | grep -E "^\[(ocr|done)\]"

In [ ]:
# CELL 6 - Cặp huấn luyện (CPU, 5-10 phút): cặp cũ (dòng / vùng có nhãn), con số của hóa đơn verified, chữ số tổng hợp.
# Mong đợi: báo cáo số cặp; "verified_receipts" khoảng 500-600 ở train
PAIRS, NUMS, SYN = f'{W}/ocr_pairs', f'{W}/number_pairs', f'{W}/synth_pairs'
!python scripts/finetune_vietocr.py build --csv "{CSV}" --ocr-dir {OCR1}/ocr_annotations --image-dir {OCR1}/preprocessed_images --split-manifest {MANIFEST} --out {PAIRS} | tail -30
!python scripts/finetune_vietocr.py numbers --ocr-dir {OCR1}/ocr_annotations --image-dir {OCR1}/preprocessed_images --split-manifest {MANIFEST} --out-numbers {NUMS} --out-synth {SYN} --synth-train 15000 --synth-val 1000 | grep -v '"/' | tail -30

In [ ]:
# CELL 7 [KIỂM THỬ] - Mốc bộ đo chữ số của VietOCR vòng 1 trên validation (2-3 phút). Mong đợi: bảng 4 nhóm
!python scripts/eval_numbers.py --pairs {PAIRS} --numbers {NUMS} --synth {SYN} --weights round1={VFT1} --out {W}/eval/numbers_round1.json | grep -E "^\|"

In [ ]:
# CELL 8 - Tinh chỉnh VietOCR từ vòng 1 với cặp cũ + chữ số tổng hợp + con số verified (khoảng 40-60 phút).
# Chọn epoch theo CER vùng CRAFT của validation (như vòng 1). Mong đợi: các dòng "[ft] epoch ..." và bảng so sánh cuối
VNUM = f'{W}/vietocr_num'
!python -u scripts/finetune_vietocr.py train --pairs {PAIRS} --out {VNUM} --init-weights {VFT1} --epochs 6 --lr 5e-5 --patience 2 --extra-pairs {SYN} --extra-pairs {NUMS}:number 2>&1 | grep -vE "it/s|Loading"

In [ ]:
# CELL 9 [KIỂM THỬ] - CỔNG 1: bộ đo chữ số, vòng 1 so với vòng chữ số. Mong đợi: bảng 8 dòng và "CỔNG 1: ĐẠT" hoặc "KHÔNG ĐẠT"
!python scripts/eval_numbers.py --pairs {PAIRS} --numbers {NUMS} --synth {SYN} --weights round1={VFT1} --weights digits={VNUM}/vietocr_finetuned.pth --out {W}/eval/numbers_compare.json | grep -E "^\|"
nr = json.load(open(f'{W}/eval/numbers_compare.json'))['results']
ft = json.load(open(f'{VNUM}/finetune_report.json'))
region_diff = ft['paired_finetuned_minus_pretrained']['region']['diff']
t1, t2 = nr['round1']['total_region'], nr['digits']['total_region']
GATE1 = t2['digits'] > t1['digits'] and t2['cer'] <= t1['cer'] and region_diff <= 0.002
print(f"TOTAL_COST vùng: chữ số đúng {100*t1['digits']:.1f}% -> {100*t2['digits']:.1f}%, CER {100*t1['cer']:.2f} -> {100*t2['cer']:.2f}")
print(f"CER vùng CRAFT validation (sau - trước): {100*region_diff:+.2f} điểm")
print("CỔNG 1:", "ĐẠT -> OCR lại và train M1b v4" if GATE1 else "KHÔNG ĐẠT -> dừng, giữ VietOCR vòng 1")

In [ ]:
# CELL 10 - (chỉ khi đạt cổng 1) OCR train + validation với VietOCR mới, rồi token dataset (khoảng 40 phút)
OCR2, TOKN = f'{W}/ocr_num', f'{W}/tok_num'
if GATE1:
    !python -u scripts/run_ocr_dataset.py {COMMON} --weights {VNUM}/vietocr_finetuned.pth --out {OCR2} 2>&1 | grep -E "^\[(ocr|done)\]"
    !python scripts/build_token_dataset.py --csv "{CSV}" --ocr-dir {OCR2}/ocr_annotations --split-manifest {MANIFEST} --out {TOKN} | head -4
    !python scripts/compare_ocr_runs.py --reference ft_n05="{TOKC}" --run num={TOKN} --out {W}/eval/ocr_num_vs_ft_n05_val.json
else:
    print("bỏ qua: không đạt cổng 1")

In [ ]:
# CELL 11 - (chỉ khi đạt cổng 1) Train M1b v4, cùng cấu hình M1b v3, trên OCR mới (khoảng 25 phút)
if GATE1:
    !python -u scripts/train_layoutxlm.py --token-dir {TOKN} --image-dir {OCR2}/preprocessed_images --out {W}/runs --name m1b_v4_num --seed 42 --class-weight sqrt_inverse --select-metric macro --epochs 20 --patience 5 2>&1 | grep -E "epoch|best|status|params"
    CK4 = one(f'{W}/runs/m1b_v4_num_*/best')
    print("CK4 =", CK4)
else:
    print("bỏ qua: không đạt cổng 1")

In [ ]:
# CELL 12 [KIỂM THỬ] - CỔNG 2: validation, M1b v4 so với M1b v3 (tổng tiền, mặt hàng). Mong đợi: bảng và "CỔNG 2: ..."
from src.evaluation.metrics import normalize_amount
from src.extraction.line_items import extract_items, check_items
def totals(path, recs):
    pred = {r['img_id']: r['pred'] for r in json.load(open(path))['per_receipt_soft_rules']}
    out = {}
    for r in recs:
        gold = {a for a in (normalize_amount(t) for t in r['gt'].get('TOTAL_COST', [])) if a}
        if gold:
            out[r['img_id']] = normalize_amount(pred[r['img_id']].get('TOTAL_COST') or '') in gold
    return out
def items(tok):
    out = {}
    for r in load_jsonl(os.path.join(tok, 'validation.jsonl')):
        gold = {a for a in (normalize_amount(t) for t in r['gt'].get('TOTAL_COST', [])) if a and a < 1e8}
        if gold:
            x = extract_items(r['words'], r['boxes']); c = check_items(x['items'], x['summary_amounts'])
            out[r['img_id']] = c['sum'] in gold or c['sum_without_discounts'] in gold
    return out
D = '--split validation --datetime --cleanup --total-from-items'
!python scripts/decode_from_probs.py --token-dir {TOKC} --probs {PROBS3} {D} --out {W}/eval/decoded_v3_val.json | grep -E "TOTAL_COST \|"
if GATE1:
    !python scripts/evaluate_layoutxlm.py --checkpoint {CK4} --token-dir {TOKN} --image-dir {OCR2}/preprocessed_images --split validation --save-probs {W}/eval/m1b_v4_val_probs.npz --out {W}/eval/m1b_v4_val.json > /dev/null
    !python scripts/decode_from_probs.py --token-dir {TOKN} --probs {W}/eval/m1b_v4_val_probs.npz {D} --out {W}/eval/decoded_v4_val.json | grep -E "TOTAL_COST \||All fields"
    a = totals(f'{W}/eval/decoded_v3_val.json', load_jsonl(os.path.join(TOKC, 'validation.jsonl')))
    b = totals(f'{W}/eval/decoded_v4_val.json', load_jsonl(os.path.join(TOKN, 'validation.jsonl')))
    ia, ib = items(TOKC), items(TOKN)
    k = sorted(set(a) & set(b)); ki = sorted(set(ia) & set(ib))
    fixed, broke = sum(b[i] and not a[i] for i in k), sum(a[i] and not b[i] for i in k)
    RES = {"total_v3": sum(a[i] for i in k), "total_v4": sum(b[i] for i in k), "n": len(k), "fixed": fixed, "broken": broke,
           "items_v3": sum(ia[i] for i in ki), "items_v4": sum(ib[i] for i in ki), "n_items": len(ki)}
    json.dump(RES, open(f'{W}/eval/gate2.json', 'w'), indent=2)
    GATE2 = RES["total_v4"] > RES["total_v3"] and fixed > broke and RES["items_v4"] >= RES["items_v3"] - 1
    print(RES)
    print("CỔNG 2:", "ĐẠT -> dùng VietOCR chữ số + M1b v4" if GATE2 else "KHÔNG ĐẠT -> giữ cấu hình cũ, ghi kết quả âm")

In [ ]:
# CELL 13 - Nén kết quả nhỏ (tab Output). Trọng số VietOCR mới và checkpoint M1b v4 nằm sẵn trong output của phiên bản.
!cd {W} && zip -q -r digit_eval.zip eval vietocr_num/finetune_report.json ocr_pairs/pairs_report.json number_pairs/number_pairs_report.json && ls -la digit_eval.zip